# YOLO and Hybrid Traffic Light Detection

In [1]:
from pathlib import Path
import csv
import json
from html import escape
from IPython.display import HTML, display

ROOT = Path.cwd()
RESULTS = ROOT / "results"
LAB_A = json.loads((ROOT.parent / "labA/assets/test_results.json").read_text(encoding="utf-8"))
BEST = "coco_yolo11s_1280"

def R(name):
    return json.loads((RESULTS / f"{name}.json").read_text(encoding="utf-8"))

def pct(value):
    return "-" if value is None else f"{value:.2f}%"

def show_table(headers, rows):
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

## 1. Objective and Setup

In Laboratory 2A we built a classical detector: MSER + Canny proposals, HOG, HSV and geometry features, and a Random Forest.
In this laboratory we solve the same problem with a ready-made YOLO model. We also test a hybrid approach: YOLO finds
the traffic light, and the classical HSV method from Laboratory 2A recognizes its color.

The task has five parts:

1. Run YOLO trained on COCO and check which traffic lights and stop signs it finds.
2. Build the hybrid approach: YOLO boxes + HSV color recognition.
3. Optional fine-tuning of a small YOLO model on five classes.
4. Compare the results with Laboratory 2A on the same test images.
5. Report precision, recall and F1, color accuracy, and error examples.

**We did not carry out the optional fine-tuning (task 3).** Without it, YOLO has no class that covers all traffic signs.
Following the task instructions for this case, we compare only traffic light detection with Laboratory 2A.
For signs, we only show what the COCO `stop sign` class can find.

### 1.1. Comparison Rules

- **Same test set:** the same 20,000 test images as in Laboratory 2A.
- **Same evaluation:** our evaluation code imports the matching and metric functions from Laboratory 2A
  (IoU >= 0.5, greedy matching by score, one prediction per reference box, duplicates count as FP).
  Detection uses one combined `traffic_light` class. Color accuracy is counted only for correctly localized lights.
- **Parameters only from train/val:** model size, input size, score threshold and HSV parameters were selected on
  1,000 validation images (`val1000`). We drew them with the same procedure and seed as in Laboratory 2A.
  The test set was used once, with frozen settings.
- **Timing:** time per image, end-to-end (image reading, network, NMS and HSV color if used), batch size 1,
  after 10 warm-up runs. GPU: NVIDIA RTX 3060 Laptop (6 GB). Laboratory 2A was measured only on CPU, so we also
  measured YOLO on CPU on the first 500 test images.

As a check, we ran our evaluation code on the saved Laboratory 2A test predictions. It reproduced the original results exactly:

In [2]:
headers = ["Detector", "TP (Lab A / our eval)", "FP (Lab A / our eval)", "FN (Lab A / our eval)", "Match"]
rows = []
for entry in LAB_A["detection"]:
    ours = R("labA_light_test" if entry["class"] == "traffic_light" else "labA_sign_test")
    keys = ("true_positive", "false_positive", "false_negative")
    rows.append([entry["class"], *[f"{entry[k]} / {ours[k]}" for k in keys], all(entry[k] == ours[k] for k in keys)])
show_table(headers, rows)
print(f"Color accuracy: Lab A {LAB_A['color']['color_accuracy_percent']:.2f}%, "
      f"our evaluation {R('labA_light_test')['color']['color_accuracy_percent']:.2f}%")

Detector,TP (Lab A / our eval),FP (Lab A / our eval),FN (Lab A / our eval),Match
traffic_sign,12014 / 12014,50964 / 50964,56978 / 56978,True
traffic_light,11773 / 11773,45443 / 45443,41044 / 41044,True


Color accuracy: Lab A 83.98%, our evaluation 83.98%


One test image (`cba8da87-947f586b.jpg`) is a truncated JPEG file. The YOLO image loader could not read it, so we read
all images with OpenCV, which decodes the available part of the file. The image is kept in the evaluation.

## 2. YOLO Trained on COCO

### 2.1. Model and Settings

We used YOLO11 models pretrained on COCO (Ultralytics), in the smallest (`yolo11n`) and small (`yolo11s`) versions.
COCO has 80 classes. Two of them are relevant: `traffic light` (class 9) and `stop sign` (class 11). Other classes are ignored.

We saved all detections with score >= 0.01 (NMS IoU 0.7, at most 100 boxes per image, FP16 on the GPU).
The final score threshold was then chosen on validation images as the threshold with the best F1, as in Laboratory 2A.

### 2.2. Model and Input Size Selection

Most traffic lights are small: the shorter side of the box is usually 8-15 pixels. YOLO resizes the image to a fixed input
size, so we expected the input size to matter. We compared two models and three input sizes on `val1000`
(1,000 images, 2,550 traffic lights):

In [3]:
headers = ["Model", "Input size", "Threshold", "Precision", "Recall", "F1", "ms / image", "FPS (GPU)"]
rows = []
for model in ("yolo11n", "yolo11s"):
    for size in (640, 960, 1280):
        r = R(f"coco_{model}_{size}_val_light")
        rows.append([model, size, f"{r['score_threshold']:.3f}", pct(r["precision_percent"]), pct(r["recall_percent"]),
                     pct(r["f1_percent"]), f"{1000 * r['mean_detection_seconds']:.1f}", f"{r['fps']:.1f}"])
show_table(headers, rows)

Model,Input size,Threshold,Precision,Recall,F1,ms / image,FPS (GPU)
yolo11n,640,0.076,22.08%,17.25%,19.37%,23.0,43.5
yolo11n,960,0.120,33.26%,25.10%,28.61%,18.2,55.0
yolo11n,1280,0.148,37.42%,30.90%,33.85%,19.4,51.6
yolo11s,640,0.097,30.88%,25.02%,27.64%,18.7,53.5
yolo11s,960,0.149,44.83%,33.14%,38.11%,18.7,53.5
yolo11s,1280,0.147,43.44%,41.92%,42.67%,18.1,55.4


![YOLO-COCO model and input size comparison on val1000](assets/B01_coco_grid.png)

*Figure 1. Left: traffic light F1 for each model and input size. Middle: F1 against speed. Right: precision-recall curves.*

A larger input size improved recall the most. For `yolo11s`, recall grew from 25.0% at 640 pixels to 41.9% at 1280 pixels,
while precision stayed at a similar level. The larger model was better at every input size.

Speed was almost the same for all variants (about 18-23 ms per image). With batch size 1, the time is dominated by image
reading and per-image overhead, not by the network size. We therefore chose **`yolo11s` with input size 1280** (validation F1 42.67%,
score threshold 0.147). All settings were frozen before testing.

![Traffic light recall by object size on val1000](assets/B02_coco_recall_by_size.png)

*Figure 2. Recall by the shorter side of the reference box.*

The largest gains are for objects between 8 and 32 pixels. Lights below 8 pixels remain difficult for every variant
(recall 14% for the selected model).

### 2.3. What the COCO Model Detects

![YOLO-COCO detections on validation images](assets/B04_coco_preview_val.png)

*Figure 3. Selected model on validation images in day, night and rain. Cyan: correct detection. Magenta: false detection.
Dashed yellow: missed reference box. Labels show the class (TL = traffic light, STOP = stop sign) and score.*

In daylight the model finds most traffic lights that are clearly visible against the sky or buildings. At night it also
detects many lights, but bright lamps and car lights cause false detections. Small and distant lights are often missed in all
conditions. Single images in full resolution are saved in `assets/examples/`.

### 2.4. Stop Signs

BDD100K has only one general `traffic sign` class. It does not say which signs are stop signs. We therefore matched the COCO
`stop sign` detections against **all** reference signs. Precision then answers the question "is a detected stop sign a traffic
sign at all?". Recall is only informative, because stop signs are a small part of all signs.

The threshold with the best F1 on `val1000` was the lowest possible score (0.01): almost every additional stop sign detection
increases the number of matched signs, while thousands of other signs remain missed.

In [4]:
s = R(f"{BEST}_stop_test")
headers = ["Split", "Reference signs", "Stop sign detections", "TP", "FP", "Precision", "Recall"]
rows = []
for name, split in ((f"{BEST}_val_stop", "val1000"), (f"{BEST}_stop_test", "test")):
    r = R(name)
    rows.append([split, r["ground_truth_objects"], r["true_positive"] + r["false_positive"], r["true_positive"],
                 r["false_positive"], pct(r["precision_percent"]), pct(r["recall_percent"])])
show_table(headers, rows)

Split,Reference signs,Stop sign detections,TP,FP,Precision,Recall
val1000,3445,191,115,76,60.21%,3.34%
test,68992,3464,2046,1418,59.06%,2.97%


![Stop sign precision as a function of score threshold](assets/B03_stop_precision.png)

*Figure 4. Stop sign precision against all BDD signs on val1000. The dashed line marks the selected threshold.*

![Stop sign detections that match a reference sign](assets/B17_stop_TP.png)

*Figure 5. Test detections that match a reference sign. High scores are real stop signs. Low-score matches include other
red or white signs, such as "Do Not Enter", and even a yellow pedestrian sign.*

![False stop sign detections](assets/B18_stop_FP.png)

*Figure 6. False detections, all with low scores: red lights, round objects and road markings.*

On the test set the model returned 3,464 stop sign detections. 59% of them matched a reference sign, but together they cover
only 3% of the 68,992 signs.

### 2.5. Why the COCO Model Cannot Detect All Traffic Signs

- **Missing classes.** The only road sign class in COCO is `stop sign`. COCO has no speed limit, warning, mandatory,
  prohibition, direction or information signs. A detector cannot return a class it was never trained on.
- **A very broad target class.** The BDD `traffic sign` class contains all sign types, also signs seen from the side or from
  behind. Even a perfect stop sign detector would cover only a small part of it.
- **A different domain.** COCO photos usually show larger objects. BDD100K images come from a car camera, with many small
  objects, night scenes, rain and motion blur.
- **No color.** COCO has a `traffic light` class, but without color. The color must be added by another method (Section 3)
  or learned by fine-tuning, which we did not carry out.

## 3. Hybrid Approach: YOLO Boxes and HSV Color

### 3.1. Method

```
image -> YOLO-COCO (traffic light, frozen input size and threshold) -> boxes -> HSV color method from Laboratory 2A
      -> traffic_light_red / yellow / green / unknown
```

For each YOLO box we used the color function from Laboratory 2A without changes. It searches the box for bright, saturated
red, yellow or green pixels and selects the strongest color if it clearly beats the others. Otherwise the result is `unknown`.

We tested three variants:

| Variant | Description | Purpose |
|---|---|---|
| **H0** | Laboratory 2A HSV parameters, unchanged | only the detector is different |
| **H1** | HSV parameters tuned on YOLO boxes from `val1000` | check whether YOLO boxes need different parameters |
| **Oracle** | color computed on the reference boxes of all test lights | upper bound for the color method alone |

For H1 we searched 486 parameter combinations on the 1,069 correctly detected validation lights: minimum saturation and
brightness, dominance ratio, hue borders between red/yellow and yellow/green, and an optional shrinking of the box by 10% on
each side. Validation accuracy increased from 70.72% (H0 parameters) to 73.15%. The changed parameters are:

In [5]:
h0 = json.loads((ROOT.parent / "labA/configs/traffic_light_colors.json").read_text(encoding="utf-8"))
h1 = json.loads((ROOT / "configs/traffic_light_colors_yolo.json").read_text(encoding="utf-8"))
show_table(["Parameter", "H0 (Laboratory 2A)", "H1 (tuned)"],
           [[k, json.dumps(h0.get(k)), json.dumps(h1.get(k))] for k in h1 if h0.get(k) != h1.get(k)])

Parameter,H0 (Laboratory 2A),H1 (tuned)
min_saturation,0.4,0.5
min_value,0.5,0.6
dominance_ratio,1.2,1.1
hue_ranges_degrees,"{""red"": [[0, 20], [340, 360]], ""yellow"": [[20, 75]], ""green"": [[75, 175]]}","{""red"": [[0, 25], [340, 360]], ""yellow"": [[25, 65]], ""green"": [[65, 175]]}"
roi_shrink,null,0.1


### 3.2. Color Results

Color is evaluated only on correctly localized lights, so each method is evaluated on a different set of lights:

In [6]:
headers = ["Method", "Correctly localized lights", "Color accuracy", "Recall red", "Recall yellow", "Recall green", "Recall unknown"]
rows = []
for name, label in (("labA_light_test", "Lab A: MSER+Canny+RF + HSV"), ("hybrid_H0_test", "YOLO-COCO + HSV (H0)"),
                    ("hybrid_H1_test", "YOLO-COCO + HSV (H1)"), ("oracle_H0_test", "Reference boxes + HSV (oracle H0)"),
                    ("oracle_H1_test", "Reference boxes + HSV (oracle H1)")):
    c = R(name)["color"]
    rows.append([label, c["correctly_localized_lights"], pct(c["color_accuracy_percent"]),
                 *[pct(100 * c["per_class"][color]["recall"]) for color in ("red", "yellow", "green", "unknown")]])
show_table(headers, rows)

Method,Correctly localized lights,Color accuracy,Recall red,Recall yellow,Recall green,Recall unknown
Lab A: MSER+Canny+RF + HSV,11773,83.98%,90.40%,50.79%,86.32%,12.15%
YOLO-COCO + HSV (H0),20301,69.60%,80.20%,67.14%,81.62%,41.56%
YOLO-COCO + HSV (H1),20301,72.94%,81.25%,54.69%,78.50%,58.20%
Reference boxes + HSV (oracle H0),52817,65.58%,76.37%,67.00%,71.45%,48.92%
Reference boxes + HSV (oracle H1),52817,65.39%,71.32%,52.53%,61.59%,66.60%


![Color confusion matrices](assets/B05_color_confusion.png)

*Figure 7. Color errors on correctly localized lights. Rows: reference; columns: prediction. Each cell shows the count and the
row percentage.*

![Color recall per class](assets/B06_color_recall_per_class.png)

*Figure 8. Color recall for each reference color.*

At first sight the hybrid is worse than Laboratory 2A (69.60% or 72.94% vs 83.98%). The main reason is the set of lights:

- YOLO finds many more lights, including many with an `unknown` reference color: 5,720 of 20,301 correct detections (28%),
  compared with 461 of 11,773 (4%) for Laboratory 2A. `unknown` is the hardest color, so the overall accuracy drops.
- On the **same lights**, detected by both methods, the accuracy is almost identical (see Section 4).
- The oracle shows that the HSV method reaches only 65.58% on all reference lights. Many of them are small, dark or seen from
  the side. Laboratory 2A mostly found bright lights, which are also easy to color.

H1 improved the accuracy on YOLO boxes by 3.3 percentage points, mainly through better recall of `unknown`. On reference boxes
H1 did not help (65.39% vs 65.58%): the tuned parameters, such as shrinking the box, fit the way YOLO draws boxes.
`yellow` remains the weakest class for every variant: precision is below 10%, mostly because yellow traffic light housings are
classified as a yellow light (Section 5).

## 4. Comparison with Laboratory 2A

### 4.1. Detection, Color and Speed

All methods were evaluated on the same 20,000 test images, at IoU >= 0.5. Thresholds were frozen on validation data.
H0 and H1 use the same YOLO boxes, so their detection results are identical.

In [7]:
cpu_seconds = lambda path, key="seconds": [json.loads(line).get(key, 0.0) for line in open(path, encoding="utf-8")]
mean = lambda values: sum(values) / len(values)
cpu_det = mean(cpu_seconds(ROOT / f"runs/predictions/{BEST}_test_cpu500.jsonl"))
headers = ["Method", "Precision", "Recall", "F1", "Color accuracy (TP)", "FPS (GPU)", "FPS (CPU)"]
a = next(e for e in LAB_A["detection"] if e["class"] == "traffic_light")
rows = [["Lab A: MSER+Canny+RF + HSV", pct(a["precision_percent"]), pct(a["recall_percent"]), pct(a["f1_percent"]),
         pct(LAB_A["color"]["color_accuracy_percent"]), "-", f"{1 / a['mean_detection_seconds']:.2f}"]]
for name, label in (("hybrid_H0_test", "Lab B: YOLO-COCO + HSV (H0)"), ("hybrid_H1_test", "Lab B: YOLO-COCO + HSV (H1)")):
    r = R(name)
    color = mean(cpu_seconds(ROOT / f"runs/predictions/{name}.jsonl", "color_seconds"))
    rows.append([label, pct(r["precision_percent"]), pct(r["recall_percent"]), pct(r["f1_percent"]),
                 pct(r["color"]["color_accuracy_percent"]), f"{r['fps']:.1f}", f"{1 / (cpu_det + color):.2f}"])
show_table(headers, rows)

Method,Precision,Recall,F1,Color accuracy (TP),FPS (GPU),FPS (CPU)
Lab A: MSER+Canny+RF + HSV,20.58%,22.29%,21.40%,83.98%,-,0.29
Lab B: YOLO-COCO + HSV (H0),40.34%,38.44%,39.37%,69.60%,37.9,6.38
Lab B: YOLO-COCO + HSV (H1),40.34%,38.44%,39.37%,72.94%,38.3,6.39


![Comparison of Laboratory 2A and the hybrid approach](assets/B07_comparison_A_vs_B.png)

*Figure 9. Left: test precision, recall and F1 for traffic lights. Right: speed on a logarithmic scale.*

![F1 against speed](assets/B08_f1_vs_fps.png)

*Figure 10. Test F1 against speed. Circles: GPU, squares: CPU.*

YOLO almost doubled precision and recall: F1 increased from 21.40% to 39.37%. It is also much faster: about 38 images per
second on the GPU and 6.4 on the CPU, compared with 0.29 for Laboratory 2A on the CPU. Even on the CPU the hybrid is about
22 times faster. The HSV color step adds about 1 ms per image.

The test GPU timing (38 FPS) is lower than on validation (55 FPS). The test predictions were collected in several resumed runs
while other jobs were using the computer, so the validation value better shows the speed of the model itself.

### 4.2. Color on the Same Traffic Lights

To compare color recognition fairly, we used only the reference lights that were correctly detected by both methods:

In [8]:
common = json.loads((RESULTS / "comparison_common_lights.json").read_text(encoding="utf-8"))
show_table(["Lab B method", "Common lights", "Lab A accuracy", "Lab B accuracy"],
           [[k.replace("GT boxes", "Reference boxes"), v["common_lights"], pct(v["first_accuracy_percent"]),
             pct(v["second_accuracy_percent"])] for k, v in common.items() if v])

Lab B method,Common lights,Lab A accuracy,Lab B accuracy
YOLO-COCO + HSV (H0),6149,83.41%,83.82%
YOLO-COCO + HSV (H1),6149,83.41%,83.77%
Reference boxes + HSV (oracle H0),11773,83.98%,84.24%
Reference boxes + HSV (oracle H1),11773,83.98%,78.65%


On the 6,149 common lights both approaches reach about 83-84%. The color method works in the same way on YOLO boxes and on
Laboratory 2A boxes. The difference in Section 3.2 comes from the different sets of detected lights.

### 4.3. Which Lights Each Method Finds

In [9]:
def matched(name):
    return {(m["image_id"], m["index"]) for m in json.loads((ROOT / f"runs/eval/{name}_matched.json").read_text())}

a_set, b_set = matched("labA_light_test"), matched("hybrid_H1_test")
total = R("labA_light_test")["ground_truth_objects"]
show_table(["Reference lights", "Only Lab A", "Only YOLO-COCO", "Both", "Neither"],
           [[total, len(a_set - b_set), len(b_set - a_set), len(a_set & b_set), total - len(a_set | b_set)]])

Reference lights,Only Lab A,Only YOLO-COCO,Both,Neither
52817,5624,14152,6149,26892


![Traffic light results by object size](assets/B09_lights_by_size.png)

*Figure 11. Test recall and precision by the shorter side of the box.*

![Traffic light results by time of day](assets/B10_lights_by_timeofday.png)

*Figure 12. Test recall and precision by time of day.*

The methods are complementary. Each finds lights the other misses, and half of all lights are missed by both.

- **Size:** YOLO is much better for lights of 16 pixels and more (recall 54-59% vs 10-22%). For lights below 8 pixels both
  methods reach only about 13% recall.
- **Time of day:** YOLO is much better in daylight (recall 44% vs 8%). At night, Laboratory 2A has a higher recall
  (38% vs 32%): its MSER proposals react strongly to bright lamps. Its precision at night is still low (21%).

![Lights detected only by Laboratory 2A](assets/B15_only_labA.png)

*Figure 13. Lights detected by Laboratory 2A but missed by YOLO: mostly bright, overexposed lamps at night, where the light
housing is not visible.*

![Lights detected only by YOLO](assets/B16_only_yolo.png)

*Figure 14. Lights detected by YOLO but missed by Laboratory 2A: mostly daytime lights, including unlit or dark housings,
pedestrian signals and lights seen from the side. YOLO recognizes the shape of the housing, not only a bright lamp.*

## 5. Error Analysis

The examples come from the test set, for the hybrid with H1 parameters. Images are random samples from each error type.
White boxes mark the analyzed object. The captions give possible causes.

### 5.1. False Detections

![False positive traffic light detections](assets/B11_errors_FP.png)

*Figure 15. False positives with their scores.*

Most false detections have low scores (0.15-0.40). We see three groups:

- **Objects similar to a light:** yellow diamond signs, street lamps and glare.
- **Lights without a reference box:** several boxes contain real lights that are not annotated, for example pedestrian
  signals or lights seen from the side or from behind. These count as errors, even if the detector is right.
- **Boxes with IoU below 0.5:** the box covers only part of the light or also the bright area around it.

### 5.2. Missed Lights

![Missed traffic lights](assets/B12_errors_FN.png)

*Figure 16. Missed lights with the shorter side of the box and time of day.*

Typical causes: small size (5-12 pixels), overexposed lamps at night where only a bright spot is visible, pedestrian signals,
and the back side of a light. COCO training images rarely contain such objects.

### 5.3. Wrong Colors

![Correctly localized lights with a wrong color](assets/B13_errors_color.png)

*Figure 17. Correctly localized lights with a wrong color, shown as reference -> prediction.*

The most common errors:

- **`unknown` -> `yellow`:** many traffic light housings in the dataset are painted yellow. The HSV method sees a large
  yellow area and returns `yellow`, even if no lamp is on.
- **`green` -> `unknown`:** overexposed green lights appear white or cyan, with low saturation.
- **Pedestrian signals:** the orange hand symbol is classified as yellow or red.
- **Two lights in one box:** the box covers neighboring lights or lamps, and the wrong one dominates.

### 5.4. Images with the Most Errors

![Test images with the most errors](assets/B14_errors_worst_images.png)

*Figure 18. Test images with the most false positives (top) and the most missed lights (bottom).*

The images with the most errors are city streets with many lights at once, by day and by night. False detections come from
car lights, street lamps, low-score boxes on lights without a reference box, and even reflections on the car hood (bottom of
the top-left and top-middle images). Missed lights are mostly long rows of small, distant lights. In such scenes one image can
contribute more than ten errors.

## 6. Conclusions

A ready-made YOLO model trained on COCO clearly improved traffic light detection over our classical method. On the same
20,000 test images, F1 increased from 21.40% to 39.37%, and the method is about 130 times faster on the GPU and 22 times
faster on the CPU. The input size mattered more than the model size, because most lights are very small.

The hybrid approach works: YOLO finds the light and the classical HSV method recognizes the color. On lights found by both
methods, color accuracy was the same as in Laboratory 2A (about 84%). The lower overall accuracy of the hybrid (72.94% with
tuned parameters) is caused by the harder lights that only YOLO finds, especially lights with an `unknown` color.
Yellow light housings and overexposed lamps remain the main color problems.

The COCO model cannot detect general traffic signs. Its only sign class is `stop sign`, which covers 3% of the BDD signs.
Fine-tuning on the five target classes, which we did not carry out, would be needed to detect all signs and to learn the
light color directly. Small objects (below 8 pixels) and night scenes remain difficult for both approaches.

We did not tune any settings on test results.